# Review round 2 check: does the Gemma judge agree with simple patterns in the executed code?
The review asked whether the judge (Gemma 4 31B, the agent's model) was validated. For 5 of the 8 targeted checks a pattern in the successfully executed code cells gives an independent verdict; it abstains when the pattern is ambiguous. Agreement is counted on runs where both give a verdict. No model calls.

In [1]:
import json, re
from collections import Counter
from pathlib import Path
import pandas as pd
REPO = Path.cwd()
while not (REPO / "quick_runs").exists(): REPO = REPO.parent
RUNS = Path("/data/anon/reveries_runs")
PROBES = sorted(p.name for p in RUNS.iterdir() if p.is_dir() and re.match(r"P1[2-7]_", p.name) and (p / "records.jsonl").exists())
LEAK = ["expenditure", "share", "active", "majorcards"]
DROP = r"(drop|exclude|remove|!=|not in|\[\s*~)"
def code_of(probe, r):
    ev = [json.loads(l) for l in open(RUNS / probe / r["task_id"] / r["run"] / "events.jsonl")]
    return "\n".join(e["code"] for e in ev if e["event"] == "code_executed" and e.get("ok"))
def rule(check, code):
    if check == "uses_class_weight":
        return bool(re.search(r"class_weight\s*=\s*['\"{]|scale_pos_weight\s*=", code)
                    or (re.search(r"class_weight\s*=\s*(?!None)\w", code) and "balanced" in code))  # tuned via a variable
    if check == "uses_stratified_split":
        return bool(re.search(r"stratify\s*=\s*[^N\s)]|Stratified(K|Shuffle)", code))
    if check == "handles_unit_suffix_in_height":
        return bool(re.search(r"(replace|extract|rstrip|strip|sub)\(.{0,40}cm|\\d\+|replace\(\s*r?['\"]\[\^?[a-zA-Z0-9.\\-]*\]", code)) if "height" in code else None
    if check == "column_x_included":  # pass = left out
        if "column_x" not in code: return False  # never named: kept by a dtype selection
        return True if re.search(DROP + r".{0,80}column_x|column_x.{0,40}" + DROP, code) else None
    if check == "used_leakage_features":  # pass = none used
        named = [c for c in LEAK if re.search(r"['\"]" + c + r"['\"]", code)]
        if not named: return False
        return True if all(re.search(DROP + r".{0,120}['\"]" + c + r"['\"]", code, re.S) for c in LEAK) else None
CHECKS = {"sf_tf01_goal_imbalance": ["uses_class_weight", "uses_stratified_split"], "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"],
          "sf_tf03_correlation_causation": ["column_x_included"], "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
rows = []
for probe in PROBES:
    for l in open(RUNS / probe / "records.jsonl"):
        r = json.loads(l)
        if r["task_id"] not in CHECKS: continue
        code = code_of(probe, r)
        for c in CHECKS[r["task_id"]]:
            rows.append({"probe": probe, "task": r["task_id"][:7], "check": c, "judge": r["checks"].get(c), "rule": rule(c, code)})
ag = pd.DataFrame(rows)
both = ag[ag.judge.notna() & ag.rule.notna()]
tab = both.assign(agree=both.judge == both.rule).groupby("check").agree.agg(["sum", "count", "mean"]).round(3)
print("probes:", len(PROBES), "| check-runs:", len(ag), "| both decided:", len(both), "| overall agreement:", round((both.judge == both.rule).mean(), 3))
tab

probes: 22 | check-runs: 550 | both decided: 490 | overall agreement: 0.996


,sum,count,mean
check,,,
column_x_included,93,94,0.989
handles_unit_suffix_in_height,84,85,0.988
used_leakage_features,91,91,1.000
uses_class_weight,110,110,1.000
uses_stratified_split,110,110,1.000


In [2]:
# Disagreements, by check and direction
dis = both[both.judge != both.rule]
print(dis.groupby(["check", "judge", "rule"]).size().to_string())
json.dump({"n_check_runs": len(ag), "n_both": len(both), "agreement": round(float((both.judge == both.rule).mean()), 4),
           "by_check": tab.reset_index().to_dict("records"), "probes": PROBES},
          open(REPO / "quick_runs/results/P14/review2_judge_check.json", "w"), indent=1)
dis.head(20)

check                          judge  rule 
column_x_included              True   False    1
handles_unit_suffix_in_height  True   False    1


,probe,task,check,judge,rule
163,P13_Ep,sf_tf02,handles_unit_suffix_in_height,True,False
416,P16_G3d,sf_tf03,column_x_included,True,False
